# FedBabu Server

> The core abstraction for `FedBabu` server: [https://openreview.net/forum?id=HuaYQfggn5u](https://openreview.net/forum?id=HuaYQfggn5u)

In [ ]:
#| default_exp servers.fedbabu

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import gc
import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs


<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_server("fedbabu")
class ServerFedBabu(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
                 

In [ ]:
#| export
@patch
def train(self: ServerFedBabu):
    res =  []
    selected_clients = self.client_selector.select()
    
    for t in range(1, self.cfg.n_rounds + 1):
        lst_active_ids = selected_clients[t-1]
        len_clients_ds = {}

        for id in lst_active_ids:
            client_state = self.state_mgr.get_state(id)
            client = self.client_fn(id= id, comm_round= t, client_state= client_state)
            client.fit()
            self.logger.info(f"Client {id} finished training.")
            self.logger.info("*"*20)
            self.state_mgr.set_state(id, client.save_state())
            
            len_clients_ds[id] = len(client.train_loader.dataset)

            del client 
            gc.collect()
            torch.cuda.empty_cache()

        self.aggregate(lst_active_ids, t, len_clients_ds)#lst_active_ids, comm_round, len_clients_ds
    

    train_res, test_res = [], []
    for id in range(self.cfg.n_clients):
        client_state = self.state_mgr.get_state(id)
        client = self.client_fn(id= id, comm_round= t, client_state= client_state)
        client.fine_tune()
        res_train = client.evaluate_local(loader= 'train')
        train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        test_res.append(res_test)
        
        del client
        gc.collect()
        torch.cuda.empty_cache()
        
    train_df, test_df = self.writer.write(lst_active_ids, train_res, test_res, t) 
    res.append((train_df, test_df))
    
    self.writer.save(res)
    self.writer.finish()

    return res

### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerFedBabu, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                if key.startswith('head'):
                    continue
                global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.load_state_dict(global_model)
        
        for id in lst_active_ids:
            client_model = self.state_mgr.get_state(id).get('model', None)
            for key in global_model.keys():
                if key.startswith('backbone'):
                    self.logger.info(f"Updating client {id} model with global backbone parameters.")
                    client_model[key] = self.model.state_dict()[key]

            self.state_mgr.set_state(id, {
                    'model': client_model,
                    'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
            })
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()